In [1]:
from google.colab import files
import os

# 1. Upload the kaggle.json file
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# 2. Create the .kaggle directory, move the token, and set secure permissions
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API successfully configured!")

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Kaggle API successfully configured!


In [2]:
!kaggle datasets download -d wordsforthewise/lending-club -f accepted_2007_to_2018Q4.csv.gz --unzip

Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
License(s): CC0-1.0
100% 374M/374M [00:07<00:00, 49.4MB/s]



In [3]:
import pandas as pd

# Load the gzipped CSV directly into memory
df = pd.read_csv(
    'accepted_2007_to_2018Q4.csv.gz',
    compression='gzip',
    on_bad_lines='skip',
    low_memory=False
)

print("Dataset successfully loaded!")
df.head()

Dataset successfully loaded!


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Filtering for terminal loan statuses (ignore 'Current', 'In Grace Period', etc.)
terminal_statuses = ["Fully Paid", "Charged Off", "Default"]
df_clean = df[df["loan_status"].isin(terminal_statuses)].copy()

In [5]:
# Creating the binary target variable (1 = Default/Charged Off, 0 = Fully Paid)
df_clean["target"] = df_clean["loan_status"].apply(lambda x: 1 if x in ["Charged Off", "Default"] else 0)

print(f"Dataset cleaned! Total terminal loans: {len(df_clean):,}")
print(f"Overall Default Rate: {df_clean['target'].mean():.2%}")

Dataset cleaned! Total terminal loans: 1,345,350
Overall Default Rate: 19.96%


Feature selection & Preventing Data Leakage

In [6]:
# Core application-time features available at origination
selected_features = [
    # Debt & Income Metrics
    "loan_amnt", "int_rate", "installment", "annual_inc", "dti",
    # Credit Bureau / Behavioral History
    "fico_range_low", "fico_range_high", "inq_last_6mths",
    "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
    "delinq_2yrs", "mths_since_last_delinq",
    # Categorical Risk Attributes
    "term", "grade", "home_ownership", "purpose", "emp_length"
]

# Create your clean modeling matrix
X = df_clean[selected_features].copy()
y = df_clean["target"].copy()

Automated Monotonic Binning & WoE Transformation

In [9]:
pip install optbinning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.8/214.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 146.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.1/354.1 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 152.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 183.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.1
    Uninstalling protobuf-7.35.1:
      Succes

In [10]:
import pandas as pd
from optbinning import BinningProcess

# Identify categorical and numerical columns automatically
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Initialize the automated binning process
# min_prebin_size=0.05 ensures no single bin represents less than 5% of the portfolio
binning_process = BinningProcess(
    variable_names=selected_features,
    categorical_variables=categorical_cols,
    min_prebin_size=0.05,
    special_codes=[-999, -9999] # Handle custom missing/error codes if present
)

# Fit the binning process to your application features and target
print("Fitting automated monotonic binning algorithms...")
binning_process.fit(X, y)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Fitting automated monotonic binning algorithms...


BinningProcess(categorical_variables=['term', 'grade', 'home_ownership',
                                      'purpose', 'emp_length'],
               special_codes=[-999, -9999],
               variable_names=['loan_amnt', 'int_rate', 'installment',
                               'annual_inc', 'dti', 'fico_range_low',
                               'fico_range_high', 'inq_last_6mths', 'open_acc',
                               'pub_rec', 'revol_bal', 'revol_util',
                               'total_acc', 'delinq_2yrs',
                               'mths_since_last_delinq', 'term', 'grade',
                               'home_ownership', 'purpose', 'emp_length'])

Extracting IV and Transforming Features

In [12]:
import pandas as pd
from optbinning import BinningProcess

# Extract the summary table containing IVs for all variables
iv_summary = binning_process.summary()
print("\n--- Top Predictive Features by Information Value (IV) ---")
print(iv_summary.sort_values("iv", ascending=False)[["name", "iv", "status"]].head(10))

# Filter variables: Keep only features with 0.02 <= IV <= 0.50
valid_features = iv_summary[
    (iv_summary["iv"] >= 0.02) &
    (iv_summary["iv"] <= 0.50) &
    (iv_summary["status"] == "OPTIMAL")
]["name"].tolist()

print(f"\nRetained {len(valid_features)} optimal features out of {len(selected_features)}.")

# Transform raw dataset into Weight of Evidence (WoE) values
X_woe_full = binning_process.transform(X, metric="woe")
X_woe = X_woe_full[valid_features]


--- Top Predictive Features by Information Value (IV) ---
               name        iv   status
1          int_rate   0.46886  OPTIMAL
16            grade  0.458427  OPTIMAL
15             term   0.17441  OPTIMAL
5    fico_range_low  0.124156  OPTIMAL
6   fico_range_high  0.124156  OPTIMAL
4               dti  0.073558  OPTIMAL
0         loan_amnt   0.03612  OPTIMAL
2       installment  0.032023  OPTIMAL
17   home_ownership  0.031303  OPTIMAL
3        annual_inc  0.029741  OPTIMAL

Retained 12 optimal features out of 20.


Training the Regulatory Logistic Regression Model

In [13]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

In [14]:
# Training Logistic Regression without regularization (to get true MLE statistical coefficients)
lr_model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
lr_model.fit(X_woe, y)

LogisticRegression(max_iter=1000, penalty=None)

In [15]:
# Extracting and validate model coefficients
intercept = lr_model.intercept_[0]
coefficients = pd.DataFrame({'Feature': valid_features,'Coefficient': lr_model.coef_[0]})

In [16]:
#Automated Regulatory Check
positive_coefs = coefficients[coefficients["Coefficient"] > 0]
if len(positive_coefs) > 0:
    print("\nWARNING: Multicollinearity detected! These features have illegal positive coefficients:")
    print(positive_coefs["Feature"].tolist())
    print("Action: Drop these features or use L1 regularization / VIF screening.")
else:
    print("\nSUCCESS: All coefficients are negative. Model is compliant with regulatory logic!")


['revol_util']
Action: Drop these features or use L1 regularization / VIF screening.


Scorecard Scaling(PDO)

In [17]:
# Defining industry standard PDO scaling parameters
target_score = 600
target_odds = 50       # 50:1 odds of Good vs Bad
pdo = 20               # 20 points doubles the odds

In [18]:
# Calculating Factor and Offset
factor = pdo / np.log(2)
offset = target_score - (factor * np.log(target_odds))
base_points = offset - (factor * intercept)

print(f"Scorecard Scaling: Factor = {factor:.2f}, Offset = {offset:.2f}")
print(f"Base Score (Starting Points for all borrowers): {base_points:.0f} points\n")

Scorecard Scaling: Factor = 28.85, Offset = 487.12
Base Score (Starting Points for all borrowers): 527 points



In [20]:
# Building the final scorecard lookup table
scorecard_rows = []

for feature in valid_features:
    # Get coefficient for this feature
    beta_i = coefficients.loc[coefficients["Feature"] == feature, "Coefficient"].values[0]

    # Extract binning summary from optbinning
    bin_table = binning_process.get_binned_variable(feature).binning_table.build()

    # Filter out totals and special rows for clean mapping
    clean_bins = bin_table[~bin_table["Bin"].isin(["Special", "Missing", "Total"])].copy()

    # Ensure 'WoE' column is numeric and handle potential NaN values
    clean_bins["WoE"] = pd.to_numeric(clean_bins["WoE"], errors='coerce')
    clean_bins["WoE"] = clean_bins["WoE"].fillna(0)

    # Calculate exact points for each bin
    # Formula: -Factor * Beta * WoE
    clean_bins["Points"] = (-factor * beta_i * clean_bins["WoE"]).round().astype(int)
    clean_bins["Feature"] = feature

    scorecard_rows.append(clean_bins[["Feature", "Bin", "WoE", "Points", "Count", "Event rate"]])

In [21]:
# Combining into a single master scorecard DataFrame
master_scorecard = pd.concat(scorecard_rows, ignore_index=True)
print("--- Sample Scorecard: Interest Rate (int_rate) ---")
print(master_scorecard[master_scorecard["Feature"] == "int_rate"][["Bin", "WoE", "Event rate", "Points"]])

--- Sample Scorecard: Interest Rate (int_rate) ---
               Bin       WoE  Event rate  Points
9     (-inf, 6.67)  1.852994    0.037634      21
10    [6.67, 7.88)  1.331262    0.061818      15
11    [7.88, 8.21)  0.976524    0.085880      11
12    [8.21, 9.73)  0.739752    0.106383       8
13   [9.73, 10.74)  0.518220    0.129352       6
14  [10.74, 11.42)  0.414811    0.141451       5
15  [11.42, 12.57)  0.242645    0.163676       3
16  [12.57, 13.18)  0.041476    0.193105       0
17  [13.18, 14.44) -0.111670    0.218093      -1
18  [14.44, 15.97) -0.251852    0.242939      -3
19  [15.97, 17.78) -0.472072    0.285690      -5
20  [17.78, 19.18) -0.685401    0.331129      -8
21  [19.18, 21.98) -0.795291    0.355903      -9
22    [21.98, inf) -1.128470    0.435360     -13
23                  0.000000    0.199650       0


Out-of-Time Scoring Pipeline

In [24]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Define X_oot and y_oot (PLACEHOLDER: Replace with actual Out-of-Time data)
X_oot = X.copy()
y_oot = y.copy()

# 1. Transform OOT validation features using the saved binning process
# Note: X_oot must contain the exact same raw feature columns as X_train
X_oot_woe_full = binning_process.transform(X_oot, metric="woe")
X_oot_woe = X_oot_woe_full[valid_features]

# 2. Predict default probabilities using the trained logistic regression model
oot_probabilities = lr_model.predict_proba(X_oot_woe)[:, 1]

# 3. Map probabilities to 300-850 credit scores using saved scaling parameters
# ln(Odds) = ln((1 - p) / p)
oot_odds = (1 - oot_probabilities) / np.clip(oot_probabilities, 1e-9, 1 - 1e-9)
oot_scores = np.round(offset + factor * np.log(oot_odds)).astype(int)

# Create a master validation DataFrame
df_val = pd.DataFrame({
    'target': y_oot,               # 1 = Default, 0 = Fully Paid
    'pd_probability': oot_probabilities,
    'credit_score': oot_scores
})

Calculating Gini Coefficient & KS-Statistic

In [25]:
def calculate_ks_gini(df, target_col, prob_col, n_bins=10):
    """
    Calculates Gini coefficient, KS statistic, and generates a regulatory decile table.
    """
    # Calculate ROC-AUC and Gini
    auc = roc_auc_score(df[target_col], df[prob_col])
    gini = 2 * auc - 1

    # Sort borrowers by predicted risk (descending Probability of Default)
    df_sorted = df.sort_values(by=prob_col, ascending=False).reset_index(drop=True)

    # Create equal-sized risk deciles
    df_sorted['decile'] = pd.qcut(df_sorted.index, q=n_bins, labels=False) + 1

    # Aggregate counts of Goods and Bads per decile
    decile_table = df_sorted.groupby('decile').agg(
        total_borrowers=(target_col, 'count'),
        bad_borrowers=(target_col, 'sum'),
        mean_pd=(prob_col, 'mean'),
        min_score=('credit_score', 'min'),
        max_score=('credit_score', 'max')
    ).reset_index()

    decile_table['good_borrowers'] = decile_table['total_borrowers'] - decile_table['bad_borrowers']

    # Calculate cumulative distributions
    total_bads = decile_table['bad_borrowers'].sum()
    total_goods = decile_table['good_borrowers'].sum()

    decile_table['cum_bads'] = decile_table['bad_borrowers'].cumsum()
    decile_table['cum_goods'] = decile_table['good_borrowers'].cumsum()

    decile_table['cum_bad_pct'] = decile_table['cum_bads'] / total_bads
    decile_table['cum_good_pct'] = decile_table['cum_goods'] / total_goods

    # KS Statistic is the absolute difference between cumulative percentage distributions
    decile_table['ks_spread'] = np.abs(decile_table['cum_bad_pct'] - decile_table['cum_good_pct']) * 100

    ks_stat = decile_table['ks_spread'].max()
    ks_decile = decile_table.loc[decile_table['ks_spread'].idxmax(), 'decile']

    print(f"--- Regulatory Discrimination Validation ---")
    print(f"Gini Coefficient: {gini:.4f} (Target >= 0.40)")
    print(f"KS Statistic    : {ks_stat:.2f}% occurring at Decile {ks_decile} (Target: 40% - 65%)\n")

    return gini, ks_stat, decile_table

# Run validation on OOT data
gini_val, ks_val, decile_summary = calculate_ks_gini(df_val, target_col='target', prob_col='pd_probability')
print(decile_summary[['decile', 'min_score', 'max_score', 'bad_borrowers', 'cum_bad_pct', 'cum_good_pct', 'ks_spread']])

--- Regulatory Discrimination Validation ---
Gini Coefficient: 0.4144 (Target >= 0.40)
KS Statistic    : 29.83% occurring at Decile 4 (Target: 40% - 65%)

   decile  min_score  max_score  bad_borrowers  cum_bad_pct  cum_good_pct  \
0       1        462        502          60773     0.226259      0.068504   
1       2        502        512          43789     0.389287      0.152782   
2       3        512        519          36279     0.524354      0.244034   
3       4        519        526          30722     0.638733      0.340447   
4       5        526        532          25955     0.735364      0.441288   
5       6        532        538          22050     0.817457      0.545755   
6       7        538        544          18312     0.885632      0.653693   
7       8        544        553          14377     0.939158      0.765286   
8       9        553        566          10601     0.978626      0.880386   
9      10        566        607           5741     1.000000      1.000000  

Calculating Population Stabiliy Index(PSI)

In [27]:
def calculate_psi(train_scores, oot_scores, n_bins=10):
    """
    Calculates Population Stability Index (PSI) across fixed score quantiles.
    """
    # Establish reference bin boundaries from the training distribution
    quantiles = np.linspace(0, 100, n_bins + 1)
    bin_edges = np.percentile(train_scores, quantiles)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    # Calculate percentage of borrowers in each bin for both cohorts
    train_counts, _ = np.histogram(train_scores, bins=bin_edges)
    oot_counts, _ = np.histogram(oot_scores, bins=bin_edges)

    # Add a tiny epsilon to prevent division by zero or log(0) errors
    epsilon = 1e-4
    train_pct = (train_counts + epsilon) / (len(train_scores) + epsilon * n_bins)
    oot_pct = (oot_counts + epsilon) / (len(oot_scores) + epsilon * n_bins)

    # Compute bin-by-bin PSI and sum across all bins
    bin_psi = (oot_pct - train_pct) * np.log(oot_pct / train_pct)
    total_psi = np.sum(bin_psi)

    # Determine regulatory stability status
    if total_psi < 0.10:
        status = "STABLE: Minimal population drift. Approved for production."
    elif total_psi < 0.25:
        status = "MODERATE DRIFT: Continue monitoring. Investigate shifting input variables."
    else:
        status = "CRITICAL DRIFT: Severe distribution shift. Immediate model retraining required."

    print(f"--- Population Stability Index (PSI) Validation ---")
    print(f"Total PSI: {total_psi:.4f}")
    print(f"Status   : {status}\n")

    # Create diagnostic table
    psi_table = pd.DataFrame({
        'Bin_Lower': bin_edges[:-1],
        'Bin_Upper': bin_edges[1:],
        'Train_Pct': train_pct * 100,
        'OOT_Pct': oot_pct * 100,
        'Bin_PSI': bin_psi
    })

    return total_psi, psi_table

# Calculate training scores first
train_probabilities = lr_model.predict_proba(X_woe)[:, 1]
train_odds = (1 - train_probabilities) / np.clip(train_probabilities, 1e-9, 1 - 1e-9)
y_train_scores = np.round(offset + factor * np.log(train_odds)).astype(int)

# Execute PSI validation comparing training scores against OOT scores
psi_stat, psi_diagnostics = calculate_psi(train_scores=y_train_scores, oot_scores=df_val['credit_score'])
print(psi_diagnostics.round(4))

--- Population Stability Index (PSI) Validation ---
Total PSI: 0.0000
Status   : STABLE: Minimal population drift. Approved for production.

   Bin_Lower  Bin_Upper  Train_Pct  OOT_Pct  Bin_PSI
0       -inf      502.0     9.6103   9.6103      0.0
1      502.0      512.0     9.6604   9.6604      0.0
2      512.0      519.0     9.4714   9.4714      0.0
3      519.0      526.0    11.0990  11.0990      0.0
4      526.0      532.0     9.9755   9.9755      0.0
5      532.0      538.0     9.9425   9.9425      0.0
6      538.0      544.0     9.2014   9.2014      0.0
7      544.0      553.0    10.6768  10.6768      0.0
8      553.0      566.0     9.9600   9.9600      0.0
9      566.0        inf    10.4026  10.4026      0.0


Calculating LOss Given Default(LGD)

In [28]:
import numpy as np
import pandas as pd

# Filter strictly for defaulted loans (where target == 1)
df_defaults = df_clean[df_clean['target'] == 1].copy()
print(f"Total defaulted loans for LGD modeling: {len(df_defaults):,}")

# Calculate net recovery amount
net_recoveries = df_defaults['recoveries'] - df_defaults['collection_recovery_fee']

# Calculate raw empirical LGD
df_defaults['lgd_raw'] = 1.0 - (net_recoveries / df_defaults['funded_amnt'])

# Strictly bound LGD within the interval [0, 1]
df_defaults['lgd_bounded'] = np.clip(df_defaults['lgd_raw'], 0.0, 1.0)

print("\n--- Empirical LGD Distribution Summary ---")
print(df_defaults['lgd_bounded'].describe().round(4))

Total defaulted loans for LGD modeling: 268,599

--- Empirical LGD Distribution Summary ---
count    268599.0000
mean          0.9372
std           0.0786
min           0.0000
25%           0.9058
50%           0.9478
75%           1.0000
max           1.0000
Name: lgd_bounded, dtype: float64


LGD Modelling

In [30]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 22.8 MB/s eta 0:00:00


In [33]:
!pip install statsmodels

import statsmodels.api as sm
from statsmodels.othermod.betareg import BetaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Apply Smithson-Verkuilen transformation for open interval (0, 1)
N = len(df_defaults)
df_defaults['lgd_beta'] = (df_defaults['lgd_bounded'] * (N - 1) + 0.5) / N

# Select continuous and categorical predictors for recovery severity
lgd_features = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'term', 'grade']

# Prepare modeling dataframe (one-hot encode categorical variables like grade and term)
X_lgd = pd.get_dummies(df_defaults[lgd_features], drop_first=True, dtype=float)
y_lgd = df_defaults['lgd_beta']

# Identify numerical features to scale (excluding one-hot encoded and 'const')
numerical_cols_to_scale = ['loan_amnt', 'int_rate', 'annual_inc', 'dti']

# Combine X and y, replace inf/-inf with NaN, and drop rows with NaN values
# This ensures X_lgd and y_lgd have matching rows after cleaning
combined_lgd = pd.concat([X_lgd, y_lgd], axis=1)
combined_lgd = combined_lgd.replace([np.inf, -np.inf], np.nan)
combined_lgd_cleaned = combined_lgd.dropna()

# Separate X_lgd and y_lgd again after cleaning
X_lgd_final = combined_lgd_cleaned.drop(columns='lgd_beta')
y_lgd_final = combined_lgd_cleaned['lgd_beta']

# Scale numerical features
scaler = StandardScaler()
X_lgd_final[numerical_cols_to_scale] = scaler.fit_transform(X_lgd_final[numerical_cols_to_scale])

# Add constant (intercept) required by statsmodels
X_lgd_final = sm.add_constant(X_lgd_final)

# Train/Test split (Chronological or random stratified split on defaults)
X_train_lgd, X_test_lgd, y_train_lgd, y_test_lgd = train_test_split(
    X_lgd_final, y_lgd_final, test_size=0.20, random_state=42
)

print("Training institutional Beta Regression model...")
# Initialize and fit BetaModel using Maximum Likelihood Estimation (MLE)
beta_model = BetaModel(endog=y_train_lgd, exog=X_train_lgd)
beta_results = beta_model.fit(maxiter=500, disp=False)

print(beta_results.summary())

Training institutional Beta Regression model...
                              BetaModel Results                               
Dep. Variable:               lgd_beta   Log-Likelihood:             7.3443e+05
Model:                      BetaModel   AIC:                        -1.469e+06
Method:            Maximum Likelihood   BIC:                        -1.469e+06
Date:                Sat, 25 Jul 2026                                         
Time:                        19:56:35                                         
No. Observations:              214822                                         
Df Residuals:                  214809                                         
Df Model:                          11                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               3.1831      0.015    205.958      0.000       3.153  

Model Evaluation using MAE and RMSE

In [34]:
# Generate out-of-sample predictions on test set
lgd_preds_beta = beta_results.predict(X_test_lgd)

# Reverse the Smithson-Verkuilen transformation to return to [0, 1] scale
lgd_preds_final = (lgd_preds_beta * N - 0.5) / (N - 1)
y_test_original = (y_test_lgd * N - 0.5) / (N - 1)

# Calculate quantitative error metrics
mae = mean_absolute_error(y_test_original, lgd_preds_final)
rmse = np.sqrt(mean_squared_error(y_test_original, lgd_preds_final))

print("--- LGD Model Out-of-Sample Performance ---")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error  : {rmse:.4f}")
print(f"Predicted Mean LGD       : {lgd_preds_final.mean():.2%}")
print(f"Actual Mean LGD          : {y_test_original.mean():.2%}")

--- LGD Model Out-of-Sample Performance ---
Mean Absolute Error (MAE): 0.0533
Root Mean Squared Error  : 0.0795
Predicted Mean LGD       : 94.03%
Actual Mean LGD          : 93.69%


Fractional Logit Regression

In [37]:
# Get the indices of the cleaned data used for BetaModel (X_lgd_final and y_lgd_final)
# These indices correspond to rows in df_defaults that had no NaNs/infs in features or lgd_beta.
cleaned_indices = X_lgd_final.index

# Use these same cleaned indices to select the corresponding 'lgd_bounded' values
# for the GLM model's endogenous variable.
y_glm_cleaned = df_defaults.loc[cleaned_indices, 'lgd_bounded']

# X_lgd_final is already cleaned, scaled, and has a constant added.
# It serves as the exogenous variable for the GLM.
X_glm_final_with_const = X_lgd_final

# Train Fractional Logit via Generalized Linear Models (Binomial family with robust standard errors)
print("Training Fractional Logit model...")
fractional_logit_model = sm.GLM(
    endog=y_glm_cleaned,
    exog=X_glm_final_with_const,
    family=sm.families.Binomial()
).fit(cov_type='HC3')

print(fractional_logit_model.summary())

Training Fractional Logit model...
                 Generalized Linear Model Regression Results                  
Dep. Variable:            lgd_bounded   No. Observations:               268528
Model:                            GLM   Df Residuals:                   268516
Model Family:                Binomial   Df Model:                           11
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -48929.
Date:                Sat, 25 Jul 2026   Deviance:                       24322.
Time:                        19:59:43   Pearson chi2:                 2.80e+04
No. Iterations:                     6   Pseudo R-squ. (CS):           0.001074
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const  

Modeling Exposure at Default (EAD) and Credit Conversion Factor (CCF)

In [38]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# ---Define EAD Strategy ---
# For revolving credit or lines of credit, calculate historical CCF on defaulted loans
# CCF = (out_prncp - initial_draw) / (revol_bal + unused_credit)

# For LendingClub installment loans, EAD is directly bounded by funded_amnt and total principal paid
# We build an EAD estimator that uses current funded amount & utilization signals

def calculate_ead(df):
    """
    Computes Exposure at Default (EAD) in dollars.
    For active/performing loans: EAD = funded_amnt (or outstanding balance if tracking dynamically).
    """
    # If modeling credit lines, apply CCF:
    # unused_credit = np.maximum(df['total_rev_hi_lim'] - df['revol_bal'], 0)
    # predicted_ccf = ccf_model.predict(X_ccf)
    # ead = df['revol_bal'] + (predicted_ccf * unused_credit)

    # For fixed installment portfolio: EAD is bounded by total funded amount at origination
    ead = df['funded_amnt'].copy()
    return ead

df_clean['ead_dollars'] = calculate_ead(df_clean)
print("EAD summary statistics ($):")
print(df_clean['ead_dollars'].describe().round(2))

EAD summary statistics ($):
count    1345350.00
mean       14411.55
std         8713.17
min          500.00
25%         8000.00
50%        12000.00
75%        20000.00
max        40000.00
Name: ead_dollars, dtype: float64


Combining PD, LGD, and EAD into Expected Loss (EL)

In [42]:
# --- Combine Model Predictions ---

# Obtain PD predictions for all loans in the portfolio using your calibrated Logistic Regression
# (X_woe contains WoE-transformed features for the full portfolio using only valid_features)
df_clean['predicted_pd'] = lr_model.predict_proba(X_woe)[:, 1]

# Obtain LGD predictions for all loans using the trained Beta Regression model
# (Transform full dataset features into the input matrix expected by statsmodels LGD engine)
X_lgd_all = pd.get_dummies(df_clean[lgd_features], drop_first=True, dtype=float)

# Scale numerical features in X_lgd_all using the SAME scaler fitted on training data
X_lgd_all[numerical_cols_to_scale] = scaler.transform(X_lgd_all[numerical_cols_to_scale])

# Add constant (intercept) and align columns with training feature set
X_lgd_all = sm.add_constant(X_lgd_all)
X_lgd_all = X_lgd_all.reindex(columns=X_train_lgd.columns, fill_value=0)

# Predict fractional LGD and reverse Smithson-Verkuilen boundary transformation
raw_lgd_preds = beta_results.predict(X_lgd_all)
N = len(df_defaults)
df_clean['predicted_lgd'] = (raw_lgd_preds * N - 0.5) / (N - 1)
df_clean['predicted_lgd'] = np.clip(df_clean['predicted_lgd'], 0.0, 1.0)

# Calculate Expected Loss (EL)
df_clean['expected_loss_dollars'] = (
    df_clean['predicted_pd'] *
    df_clean['predicted_lgd'] *
    df_clean['ead_dollars']
)

df_clean['expected_loss_pct'] = df_clean['predicted_pd'] * df_clean['predicted_lgd']

In [43]:
# --- Portfolio Aggregation & Risk Metrics ---
total_portfolio_exposure = df_clean['ead_dollars'].sum()
total_expected_loss = df_clean['expected_loss_dollars'].sum()
portfolio_el_rate = (total_expected_loss / total_portfolio_exposure) * 100

print("=" * 50)
print("       PORTFOLIO EXPECTED LOSS (EL) REPORT       ")
print("=" * 50)
print(f"Total Portfolio Loans    : {len(df_clean):,}")
print(f"Total Portfolio Exposure : ${total_portfolio_exposure:,.2f}")
print(f"Total Expected Loss ($)  : ${total_expected_loss:,.2f}")
print(f"Portfolio EL Rate        : {portfolio_el_rate:.2f}%")
print("-" * 50)
print(f"Average Predicted PD     : {df_clean['predicted_pd'].mean():.2%}")
print(f"Average Predicted LGD    : {df_clean['predicted_lgd'].mean():.2%}")
print("=" * 50)

       PORTFOLIO EXPECTED LOSS (EL) REPORT       
Total Portfolio Loans    : 1,345,350
Total Portfolio Exposure : $19,388,585,275.00
Total Expected Loss ($)  : $3,900,172,837.99
Portfolio EL Rate        : 20.12%
--------------------------------------------------
Average Predicted PD     : 19.97%
Average Predicted LGD    : 94.22%


Expected Loss (EL) vs. Unexpected Loss (UL) and Stress Testing

In [44]:
def stress_test_portfolio(df, pd_multiplier=1.35, lgd_additive_shift=0.10):
    """
    Simulates macroeconomic downturn scenario (e.g., severe recession):
    - PDs increase by multiplier (e.g., +35% risk across all buckets)
    - LGDs increase due to depressed asset recovery values (e.g., +10% absolute loss)
    """
    stressed_df = df.copy()

    # Apply macroeconomic stress multipliers
    stressed_df['stressed_pd'] = np.clip(stressed_df['predicted_pd'] * pd_multiplier, 0, 1)
    stressed_df['stressed_lgd'] = np.clip(stressed_df['predicted_lgd'] + lgd_additive_shift, 0, 1)

    # Recalculate Stressed Expected Loss
    stressed_df['stressed_el_dollars'] = (
        stressed_df['stressed_pd'] *
        stressed_df['stressed_lgd'] *
        stressed_df['ead_dollars']
    )

    baseline_el = df['expected_loss_dollars'].sum()
    stressed_el = stressed_df['stressed_el_dollars'].sum()
    pct_increase = ((stressed_el - baseline_el) / baseline_el) * 100

    print("\n--- MACROECONOMIC STRESS TEST SCENARIO ---")
    print(f"Baseline Expected Loss ($) : ${baseline_el:,.2f}")
    print(f"Stressed Expected Loss ($) : ${stressed_el:,.2f}")
    print(f"Capital Reserve Shortfall  : +${stressed_el - baseline_el:,.2f} (+{pct_increase:.1f}%)")

    return stressed_df

# Example: Run severe downturn stress test scenario
stressed_portfolio = stress_test_portfolio(df_clean, pd_multiplier=1.40, lgd_additive_shift=0.12)


--- MACROECONOMIC STRESS TEST SCENARIO ---
Baseline Expected Loss ($) : $3,900,172,837.99
Stressed Expected Loss ($) : $5,813,691,071.16
Capital Reserve Shortfall  : +$1,913,518,233.17 (+49.1%)
